# 라이브러리 호출

In [ ]:
from collections import defaultdict
import numpy as np
import pandas as pd
import csv
import json
import logging
from logging import Logger
import os
import sys
from typing import Callable, Dict, List, Tuple, Literal, Union
from functools import reduce
from copy import deepcopy

from rdkit import Chem

from tensorboardX import SummaryWriter
import torch
import torch.nn as nn
from torch.optim import Optimizer
from torch.optim.lr_scheduler import _LRScheduler, ExponentialLR
from tqdm import tqdm, trange

import chemprop
from chemprop.args import TrainArgs, PredictArgs, HyperoptArgs
from chemprop.data import get_task_names, get_data, validate_dataset_type, MoleculeDataset, MoleculeDataLoader
from chemprop.train import run_training, train, predict
from chemprop.constants import TEST_SCORES_FILE_NAME, TRAIN_LOGGER_NAME, MODEL_FILE_NAME, HYPEROPT_LOGGER_NAME
from chemprop.utils import create_logger, makedirs, load_args, update_prediction_args, load_checkpoint, load_scalers

# 모델 학습 환경 설정

In [ ]:
args = TrainArgs()

# 1. 필수 인자만 먼저 전달 (split_sizes는 여기서 건드리지 않습니다)
args = args.parse_args([
    '--data_path', '../../Data/ML_data/Data_split/train.csv',
    '--dataset_type', 'classification',
    '--save_dir', '../../Results/Trained_model/DMPNN_RN_Ensemble_5'
])

# 2. 데이터 경로 및 상세 설정 (이 단계에서 separate_val_path가 할당되어야 합니다)
args.smiles_columns = ['SMILES']
args.gpu = 1
args.features_path = ["../../Data/ML_data/Data_split/Features/RN_train.npz"]
args.no_features_scaling = True
args.target_columns = ['G1','G2','G3','G4','G5','G6','G7','G8','G9','G10','G11','G12','G13']

# 별도 경로 설정
args.separate_val_path = "../../Data/ML_data/Data_split/val.csv"
args.separate_test_path = "../../Data/ML_data/Data_split/test.csv"
args.separate_val_features_path = ["../../Data/ML_data/Data_split/Features/RN_val.npz"]
args.separate_test_features_path = ["../../Data/ML_data/Data_split/Features/RN_test.npz"]

# 3. [중요] 모든 경로를 넣은 후, 마지막에 split_sizes를 조정합니다.
args.split_sizes = [1.0, 0.0, 0.0] 

# 4. 기타 모델 설정
args.config_path = "../../Data/ML_data/Data_split/Opt_hyperpars/DMPNN_RN.json"
args.ensemble_size = 5
args.bond_features_path = None

# 5. 최종 프로세스 실행
args.process_args()

# 데이터 불러오기

In [ ]:
# logger: quite and verbose file
logger = create_logger(name=TRAIN_LOGGER_NAME, save_dir=args.save_dir, quiet=args.quiet)
if logger is not None:
    debug, info = logger.debug, logger.info
else:
    debug = info = print

# Initialize relevant variables
init_seed = args.seed
save_dir = args.save_dir
args.task_names = get_task_names(path=args.data_path, smiles_columns=args.smiles_columns,
                                 target_columns=args.target_columns, ignore_columns=args.ignore_columns)


# Print args
# debug('Args')
# debug({key:value for key, value in args.__dict__.items() if not key.startswith('_') and not callable(key)})

# Save args
makedirs(args.save_dir)
#     args.save(os.path.join(args.save_dir, 'args.json'))

# Get data
debug('Loading data')
data = get_data(
    path=args.data_path,
    args=args,
    smiles_columns=args.smiles_columns,
    logger=logger,
    skip_none_targets=True
)
validate_dataset_type(data, dataset_type=args.dataset_type)
args.features_size = data.features_size()

if args.atom_descriptors == 'descriptor':
    args.atom_descriptors_size = data.atom_descriptors_size()
    args.ffn_hidden_size += args.atom_descriptors_size
elif args.atom_descriptors == 'feature':
    args.atom_features_size = data.atom_features_size()
    set_extra_atom_fdim(args.atom_features_size)
if args.bond_features_path is not None:
    args.bond_features_size = data.bond_features_size()
    set_extra_bond_fdim(args.bond_features_size)

debug(f'Number of tasks = {args.num_tasks}')

# 모델 학습

In [ ]:
# chemprop cross_validate
# Run training on different random seeds for each fold
all_scores = defaultdict(list)
for fold_num in range(args.num_folds):
    info(f'Fold {fold_num}')
    args.seed = init_seed + fold_num
    args.save_dir = os.path.join(save_dir, f'fold_{fold_num}')
    makedirs(args.save_dir)
    data.reset_features_and_targets()
    model_scores = run_training(args, data, logger)
    for metric, scores in model_scores.items():
        all_scores[metric].append(scores)
all_scores = dict(all_scores)    

In [ ]:
model_scores

In [ ]:
# Convert scores to numpy arrays
for metric, scores in all_scores.items():
    all_scores[metric] = np.array(scores)

# Report results
info(f'{args.num_folds}-fold cross validation')

# Report scores for each fold
for fold_num in range(args.num_folds):
    for metric, scores in all_scores.items():
        info(f'\tSeed {init_seed + fold_num} ==> test {metric} = {np.nanmean(scores[fold_num]):.6f}')

        if args.show_individual_scores:
            for task_name, score in zip(args.task_names, scores[fold_num]):
                info(f'\t\tSeed {init_seed + fold_num} ==> test {task_name} {metric} = {score:.6f}')

# Report scores across folds
for metric, scores in all_scores.items():
    avg_scores = np.nanmean(scores, axis=1)  # average score for each model across tasks
    mean_score, std_score = np.nanmean(avg_scores), np.nanstd(avg_scores)
    info(f'Overall test {metric} = {mean_score:.6f} +/- {std_score:.6f}')

    if args.show_individual_scores:
        for task_num, task_name in enumerate(args.task_names):
            info(f'\tOverall test {task_name} {metric} = '
                 f'{np.nanmean(scores[:, task_num]):.6f} +/- {np.nanstd(scores[:, task_num]):.6f}')

# Save scores
with open(os.path.join(save_dir, TEST_SCORES_FILE_NAME), 'w') as f:
    writer = csv.writer(f)

    header = ['Task']
    for metric in args.metrics:
        header += [f'Mean {metric}', f'Standard deviation {metric}'] + \
                  [f'Fold {i} {metric}' for i in range(args.num_folds)]
    writer.writerow(header)

    for task_num, task_name in enumerate(args.task_names):
        row = [task_name]
        for metric, scores in all_scores.items():
            task_scores = scores[:, task_num]
            mean, std = np.nanmean(task_scores), np.nanstd(task_scores)
            row += [mean, std] + task_scores.tolist()
        writer.writerow(row)

# Determine mean and std score of main metric
avg_scores = np.nanmean(all_scores[args.metric], axis=1)
mean_score, std_score = np.nanmean(avg_scores), np.nanstd(avg_scores)
print(mean_score, std_score)

# 특정 화합물 예측

In [ ]:
args = PredictArgs()

args.smiles_columns: List[str] = ['SMILES']   
args.gpu: int = 1
args.checkpoint_dir = "../../Results/Trained_model/DMPNN_RN_Ensemble_5/fold_0/"
args.test_path: str = "../../Data/Mtb_inhibitors/Mtb_inhibitors.csv"
args.features_path: List[str] = ["../../Data/Mtb_inhibitors/Features/RN_Mtb_inhibitors.npz"]
args.preds_path: str = "../../Results/Mtb_inhibitors_pred/Mtb_inhibitors_DMPNN_preds.csv"
args.no_features_scaling = True

args.process_args()

In [ ]:
# chemprop make_predictions

print('Loading training args')
train_args = load_args(args.checkpoint_paths[0])

num_tasks, task_names = train_args.num_tasks, train_args.task_names

update_prediction_args(predict_args=args, train_args=train_args)
args: Union[PredictArgs, TrainArgs]

if args.atom_descriptors == 'feature':
    set_extra_atom_fdim(train_args.atom_features_size)

if args.bond_features_path is not None:
    set_extra_bond_fdim(train_args.bond_features_size)

print('Loading data')
full_data = get_data(path=args.test_path, smiles_columns=args.smiles_columns, target_columns=[], ignore_columns=[],
                         skip_invalid_smiles=False, args=args, store_row=not args.drop_extra_columns)

print('Validating SMILES')
full_to_valid_indices = {}
valid_index = 0
for full_index in range(len(full_data)):
    if all(mol is not None for mol in full_data[full_index].mol):
        full_to_valid_indices[full_index] = valid_index
        valid_index += 1

test_data = MoleculeDataset([full_data[i] for i in sorted(full_to_valid_indices.keys())])

print(f'Test size = {len(test_data):,}')

# Predict with each model individually and sum predictions
if args.dataset_type == 'multiclass':
    sum_preds = np.zeros((len(test_data), num_tasks, args.multiclass_num_classes))
else:
    sum_preds = np.zeros((len(test_data), num_tasks))

# Create data loader
test_data_loader = MoleculeDataLoader(
    dataset=test_data,
    batch_size=args.batch_size,
    num_workers=args.num_workers
)

# Partial results for variance robust calculation.
if args.ensemble_variance:
    all_preds = np.zeros((len(test_data), num_tasks, len(args.checkpoint_paths)))

print(f'Predicting with an ensemble of {len(args.checkpoint_paths)} models')
for index, checkpoint_path in enumerate(tqdm(args.checkpoint_paths, total=len(args.checkpoint_paths))):
    # Load model and scalers
    model = load_checkpoint(checkpoint_path, device=args.device)
    scaler, features_scaler, atom_descriptor_scaler, bond_feature_scaler, _ = load_scalers(checkpoint_path)

    # Normalize features
    if args.features_scaling or train_args.atom_descriptor_scaling or train_args.bond_feature_scaling:
        test_data.reset_features_and_targets()
        if args.features_scaling:
            test_data.normalize_features(features_scaler)
        if train_args.atom_descriptor_scaling and args.atom_descriptors is not None:
            test_data.normalize_features(atom_descriptor_scaler, scale_atom_descriptors=True)
        if getattr(train_args, 'bond_feature_scaling', False) and getattr(args, 'bond_features_size', 0) > 0:
                test_data.normalize_features(bond_feature_scaler, scale_bond_features=True)

    # Make predictions
    model_preds = predict(
        model=model,
        data_loader=test_data_loader,
        scaler=scaler
    )
    sum_preds += np.array(model_preds)
    if args.ensemble_variance:
        all_preds[:, :, index] = model_preds

# Ensemble predictions
avg_preds = sum_preds / len(args.checkpoint_paths)
avg_preds = avg_preds.tolist()

if args.ensemble_variance:
    all_epi_uncs = np.var(all_preds, axis=2)
    all_epi_uncs = all_epi_uncs.tolist()

# Save predictions
print(f'Saving predictions to {args.preds_path}')
assert len(test_data) == len(avg_preds)
if args.ensemble_variance:
    assert len(test_data) == len(all_epi_uncs)
makedirs(args.preds_path, isfile=True)

# Get prediction column names
if args.dataset_type == 'multiclass':
    task_names = [f'{name}_class_{i}' for name in task_names for i in range(args.multiclass_num_classes)]
else:
    task_names = task_names

# Copy predictions over to full_data
for full_index, datapoint in enumerate(full_data):
    valid_index = full_to_valid_indices.get(full_index, None)
    preds = avg_preds[valid_index] if valid_index is not None else ['Invalid SMILES'] * len(task_names)
    if args.ensemble_variance:
        epi_uncs = all_epi_uncs[valid_index] if valid_index is not None else ['Invalid SMILES'] * len(task_names)

    # If extra columns have been dropped, add back in SMILES columns
    if args.drop_extra_columns:
        datapoint.row = OrderedDict()

        smiles_columns = args.smiles_columns

        for column, smiles in zip(smiles_columns, datapoint.smiles):
            datapoint.row[column] = smiles

    # Add predictions columns
    if args.ensemble_variance:
        for pred_name, pred, epi_unc in zip(task_names, preds, epi_uncs):
            datapoint.row[pred_name] = pred
            datapoint.row[pred_name+'_epi_unc'] = epi_unc
    else:
        for pred_name, pred in zip(task_names, preds):
            datapoint.row[pred_name] = pred

# Save
with open(args.preds_path, 'w') as f:
    writer = csv.DictWriter(f, fieldnames=full_data[0].row.keys())
    writer.writeheader()

    for datapoint in full_data:
        writer.writerow(datapoint.row)


# 13개 결핵균 유전자 클러스터별 임계값 기준 특정 화합물 저해 효과 예측

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc
from chemprop.utils import load_checkpoint
from chemprop.data import MoleculeDataLoader, get_data

# --- [설정부] ---
BASE_RESULT_PATH = '..\\..\\Results\\Trained_model\\DMPNN_RN_Ensemble_5\\fold_0'
MODEL_PATHS = [os.path.join(BASE_RESULT_PATH, f'model_{i}', 'model.pt') for i in range(5)]
CUTOFF_FILE = ''  # 파일이 있다면 파일명을 넣으세요.
PRED_FILE_IN_PATH = '..\\..\\Data\\Mtb_inhibitors\\'
PRED_FILE_OUT_PATH = '..\\..\\Results\\Mtb_inhibitors_pred\\'

# 13개 유전자 클러스터 이름 정의
TASK_NAMES = [f'C{i}' for i in range(1, 14)]

def load_optimal_cutoffs(file_path=None):
    """임계값을 로드하거나 기본값을 반환합니다."""
    paper_cutoffs = {
        'C1': 0.04, 'C2': 0.02, 'C3': 0.05, 'C4': 0.06, 'C5': 0.04,
        'C6': 0.05, 'C7': 0.08, 'C8': 0.05, 'C9': 0.02, 'C10': 0.03,
        'C11': 0.07, 'C12': 0.05, 'C13': 0.06
    }

    if file_path and os.path.exists(file_path):
        try:
            df = pd.read_csv(file_path)
            df['Cluster_ID'] = df['Cluster'].apply(lambda x: x.replace('Cluster ', 'C') if 'Cluster' in x else x)
            file_cutoffs = dict(zip(df['Cluster_ID'], df['Cutoff'].astype(float)))
            print(f"✅ 파일 '{file_path}'에서 임계값을 성공적으로 로드했습니다.")
            return file_cutoffs
        except Exception as e:
            print(f"⚠️ 파일 로드 중 오류 발생: {e}. 기본값을 사용합니다.")
    else:
        print("ℹ️ 임계값 파일이 없거나 지정되지 않았습니다. 기본 값을 사용합니다.")
    
    return paper_cutoffs

CUTOFF_DICT = load_optimal_cutoffs(CUTOFF_FILE)

def get_ensemble_predictions_logic(model_paths, dataset, args):
    """5개 모델의 앙상블 예측 수행 및 평균값 반환"""
    data_loader = MoleculeDataLoader(dataset=dataset, batch_size=args.batch_size, num_workers=0)
    all_preds = []
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    for path in model_paths:
        model = load_checkpoint(path, device=device)
        model.eval()
        m_preds = []
        with torch.no_grad():
            for batch in data_loader:
                batch_preds = model(batch.batch_graph(), batch.features())
                m_preds.append(batch_preds.cpu().numpy())
        all_preds.append(np.concatenate(m_preds, axis=0))
    
    return np.mean(all_preds, axis=0)

def predict_and_analyze(input_file_name, output_csv='Mtb_inhibitors_DMPNN_preds_active_results.csv'):
    """신규 물질에 대한 저해 활성 예측 및 시각화"""
    full_input_path = os.path.join(PRED_FILE_IN_PATH, input_file_name)
    df_new = pd.read_csv(full_input_path)
    
    # SMILES 컬럼 탐색
    smiles_col = next((col for col in df_new.columns if col.lower() == 'smiles'), None)
    if smiles_col is None:
        raise KeyError(f"파일에 'smiles' 컬럼이 없습니다. 현재 컬럼: {df_new.columns.tolist()}")
    
    print(f"🔍 '{smiles_col}' 컬럼 인식 완료. 앙상블 예측 진행 중... (총 {len(df_new)}개)")

    # [핵심 수정] 정답이 없는 데이터이므로 target_columns=[] 로 설정하여 ValueError 방지
    test_data = get_data(
        path=full_input_path, 
        args=args, 
        smiles_columns=[smiles_col],
        target_columns=[],  # 신규 예측 시에는 타겟 컬럼 검사를 무시함
        skip_none_targets=False # 타겟이 없어도 데이터를 로드함
    )
    
    # 앙상블 예측 수행
    avg_scores = get_ensemble_predictions_logic(MODEL_PATHS, test_data, args)

    # 결과 판정 및 저장
    for i, task in enumerate(TASK_NAMES):
        cutoff = CUTOFF_DICT.get(task, 0.05)
        df_new[f'{task}_Prob'] = avg_scores[:, i]
        df_new[f'{task}_Active'] = (avg_scores[:, i] >= cutoff).astype(int)
    
    df_new.to_csv(PRED_FILE_OUT_PATH + output_csv, index=False)
    print(f"✅ 예측 완료 및 저장: {output_csv}")

    # 결과 시각화
    plt.figure(figsize=(14, 7))
    prob_cols = [f'{task}_Prob' for task in TASK_NAMES]
    sns.boxplot(data=df_new[prob_cols])
    
    for i, task in enumerate(TASK_NAMES):
        plt.hlines(CUTOFF_DICT.get(task, 0.05), xmin=i-0.4, xmax=i+0.4, colors='red', linestyles='--')

    plt.title("Prediction Probability Distribution with Optimal Cutoffs (Red Lines)")
    plt.xticks(range(len(TASK_NAMES)), TASK_NAMES, rotation=45)
    plt.ylabel("Probability")
    plt.tight_layout()
    plt.show()

# --- 실행 부분 ---
# 1. 신규 물질 예측
predict_and_analyze('Mtb_inhibitors.csv')

# 모델 기반 결핵균 저해 효과 예상 후보 화합물 초기 스크리닝 서비스

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from rdkit import Chem
from rdkit.Chem import Draw
import os
from IPython.display import display

# --- [1. COG_MAPPING (전문 국문 명칭)] ---
COG_MAPPING = {
    'C1':  {'Name': 'TCA 회로 및 중심 탄소 대사'},
    'C2':  {'Name': '번역 및 tRNA 합성'},
    'C3':  {'Name': '비방향족 아미노산 생합성'},
    'C4':  {'Name': '방향족 아미노산 생합성'},
    'C5':  {'Name': '스트레스 방어 및 대사 경로 평형 유지'},
    'C6':  {'Name': 'tRNA 및 초기 폴리펩타이드 변형'},
    'C7':  {'Name': '엽산 및 퓨린 대사'},
    'C8':  {'Name': 'DNA 복제 및 전사 조절'},
    'C9':  {'Name': '지질 및 지질 보조인자 합성'}, 
    'C10': {'Name': '당분해 및 당신생합성'},
    'C11': {'Name': '펩티도글리칸 및 전구체 합성'},
    'C12': {'Name': '퓨린 및 피리미딘 대사'},
    'C13': {'Name': '포르피린 화합물 합성'}
}

# --- [2. 시각화 함수] ---
def visualize_candidate(row, cutoff_dict):
    smiles = row['SMILES']
    name = row.get('Name', 'Unknown_ID')
    mol = Chem.MolFromSmiles(smiles)
    img = Draw.MolToImage(mol, size=(300, 300))
    
    categories = [f'C{i}' for i in range(1, 14)]
    values = [row[f'{c}_Prob'] for c in categories]
    cutoffs = [cutoff_dict.get(c, 0.05) for c in categories]
    
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=values, theta=categories, fill='toself', name='저해 확률', line=dict(color='royalblue')))
    fig.add_trace(go.Scatterpolar(r=cutoffs, theta=categories, mode='lines', line=dict(color='red', dash='dash'), name='임계값'))

    for c_id in categories:
        fig.add_trace(go.Scatterpolar(r=[None], theta=[None], mode='markers', name=f"{c_id}: {COG_MAPPING[c_id]['Name']}", marker=dict(color='rgba(0,0,0,0)')))
    
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1]), angularaxis=dict(direction='clockwise')),
        showlegend=True,
        legend=dict(title_text="<b>Cluster Descriptions</b>", yanchor="top", y=1, xanchor="left", x=1.1, font=dict(size=10)),
        title=f"Antibacterial Profile: {name} (Hits: {int(row['Hit_Count'])})",
        margin=dict(r=150)
    )
    return img, fig

# --- [3. 메인 프로세스 실행 부분] ---
PRED_FILE_OUT_PATH = '..\\..\\Results\\Mtb_inhibitors_pred\\'
RESULTS_FILE = os.path.join(PRED_FILE_OUT_PATH, 'Mtb_inhibitors_DMPNN_preds_active_results.csv')
CUTOFF_DICT = {'C1': 0.04, 'C2': 0.02, 'C3': 0.05, 'C4': 0.06, 'C5': 0.04,
               'C6': 0.05, 'C7': 0.08, 'C8': 0.05, 'C9': 0.02, 'C10': 0.03,
               'C11': 0.07, 'C12': 0.05, 'C13': 0.06}

def get_mechanism_profile(row):
    """구분자를 '|' 대신 줄바꿈('\n')으로 변경하여 반환"""
    active_clusters = [f'C{i}' for i in range(1, 14) if row.get(f'C{i}_Active') == 1]
    mechs = [f"{c}: {COG_MAPPING[c]['Name']}" for c in active_clusters if c in COG_MAPPING]
    return "\n".join(mechs) if mechs else "Active 없음"

def generate_expert_summary(row):
    hit_count = int(row['Hit_Count'])
    # 요약 문구에서는 첫 번째 줄의 기전만 참조
    primary_mech = row['Mechanism_Profile'].split('\n')[0].strip() if hit_count > 0 else "N/A"
    return (f"이 화합물은 {hit_count}개의 필수 클러스터를 저해할 것으로 예상됩니다.")

# 분석 데이터 로드
df = pd.read_csv(RESULTS_FILE)

# --- [표 출력 1] 예측 결과 원본 전체 내용 출력 ---
print("\n" + "="*80)
print("1. 예측 원본 데이터 리포트 (Mtb_inhibitors_DMPNN_preds_active_results.csv)")
print("="*80)
display(df)

# 필터링 및 랭킹 작업
df['Hit_Count'] = df[[f'C{i}_Active' for i in range(1, 14)]].sum(axis=1)
report_df = df[df['Hit_Count'] > 0].sort_values(by='Hit_Count', ascending=False).reset_index(drop=True)

if not report_df.empty:
    report_df['Mechanism_Profile'] = report_df.apply(get_mechanism_profile, axis=1)
    report_df['Expert_Summary'] = report_df.apply(generate_expert_summary, axis=1)

    # --- [리포트 요약 구성] ---
    summary_cols = ['Name', 'SMILES', 'Hit_Count', 'Mechanism_Profile', 'Expert_Summary']
    final_summary_df = report_df[summary_cols]

    # --- [파일 저장] 요약 리포트 내용 저장 ---
    csv_save_path = os.path.join(PRED_FILE_OUT_PATH, 'Mtb_inhibitors_DMPNN_Screening_Report.csv')
    final_summary_df.to_csv(csv_save_path, index=False, encoding='utf-8-sig')
    
    try:
        excel_save_path = os.path.join(PRED_FILE_OUT_PATH, 'Mtb_inhibitors_DMPNN_Screening_Report.xlsx')
        final_summary_df.to_excel(excel_save_path, index=False)
        print(f"\n📊 요약 리포트 저장 완료:\n - CSV: {csv_save_path}\n - Excel: {excel_save_path}")
    except Exception as e:
        print(f"⚠️ Excel 저장 중 오류: {e}")

    # --- [표 출력 2] 필터링된 요약 리포트 출력 ---
    print("\n" + "="*80)
    print("2. 스크리닝 요약 리포트 (필터링 및 메커니즘 분석 결과)")
    print("="*80)

    # 1. 컬럼별 너비 정의 (SMILES 길이를 줄이기 위함)
    # final_summary_df의 컬럼 인덱스를 기준으로 스타일을 적용합니다.
    # Name은 보통 짧으므로 너비를 줄이고, SMILES에 명시적인 제한을 둡니다.
    
    styled_summary = final_summary_df.style.set_properties(**{
        'white-space': 'pre-wrap',      # 줄바꿈 허용
        'text-align': 'left',          # 왼쪽 정렬
        'vertical-align': 'top',       # 상단 정렬
        'padding': '8px 10px',         # 여백 최적화
        'border': '1px solid #eeeeee'  # 연한 테두리
    }).set_table_styles([
        # 헤더 스타일
        {'selector': 'th', 'props': [
            ('background-color', '#f8f9fa'), 
            ('color', '#333333'),
            ('font-weight', 'bold'),
            ('text-align', 'center'),
            ('border', '1px solid #dddddd')
        ]},
        # SMILES 컬럼(2번째 컬럼) 너비 강제 제한
        {'selector': 'td.col1', 'props': [
            ('max-width', '300px'),    # SMILES 최대 너비 지정
            ('word-break', 'break-all') # 긴 문자열 강제 줄바꿈
        ]},
        # Name 컬럼(1번째 컬럼) 너비 제한
        {'selector': 'td.col0', 'props': [
            ('max-width', '150px')
        ]},
        # 테이블 전체 레이아웃
        {'selector': 'table', 'props': [('width', 'auto'), ('table-layout', 'fixed')]}
    ])
    # 인덱스 숨기기 및 출력
    display(styled_summary.hide(axis='index'))

    # 3. 개별 화합물 분석 루프
    for idx, row in report_df.iterrows():
        compound_name = str(row.get('Name', f'Compound_{idx}'))
        print(f"\n▶ [Rank {idx+1}] 상세 분석: {compound_name}")
        
        mol_img, radar_fig = visualize_candidate(row, CUTOFF_DICT)
        
        img_path = os.path.join(PRED_FILE_OUT_PATH, f"{compound_name}.png")
        mol_img.save(img_path)
        
        display(mol_img)
        radar_fig.show(renderer="notebook")
        print(f"✅ {compound_name}: 이미지 저장 및 차트 생성 완료")

else:
    print("유효한 후보 물질을 찾지 못했습니다.")